In [3]:
import os
import io
import ast
import fitz
import json
import torch
import base64
import requests
import numpy as np
import pandas as pd
from PIL import Image
from dotenv import load_dotenv
from FlagEmbedding import LayerWiseFlagLLMReranker


from colpali_engine.models import ColQwen2, ColQwen2Processor

import psycopg2
from pgvector.psycopg2 import register_vector

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.messages import HumanMessage, SystemMessage

c:\Users\UserAdmin\Documents\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
load_dotenv()

True

In [100]:
JINA_API_KEY = os.getenv("JINA_API_KEY")

In [328]:
with open("idf_map.json", "r") as f:
    idf_map = json.load(f)

In [329]:
idf_map

{'a': 0.578077850775158,
 'guide': 0.7691330875378674,
 'gov': 0.4554755286828257,
 'into': 1.9218125974762528,
 'vendors': 0.24783616390458127,
 'image': 0.13005312824819776,
 'login': 0.5355182363563621,
 'log': 1.3156767939059373,
 'portal': 3.0204248861443626,
 'to': 0.2795848622191615,
 'user': 1.074514737089049,
 'this': 0.7691330875378674,
 'e': 0.7178397931503168,
 'individual': 1.148622709242771,
 'without': 1.5163474893680884,
 'singpass': 0.8232003088081431,
 'applicable': 0.7178397931503168,
 'or': 1.074514737089049,
 'sole': 1.7676619176489945,
 'pas': 2.327277705584417,
 'as': 0.8232003088081431,
 'type': 2.327277705584417,
 'singpas': 2.327277705584417,
 'societies': 1.7676619176489945,
 'password': 1.410986973710262,
 'method': 0.9409833444645266,
 'personal': 1.410986973710262,
 'for': 0.050010420574661416,
 '1': 0.9409833444645266,
 'payment': 1.410986973710262,
 'word': 2.327277705584417,
 'entity': 0.8803587226480917,
 'but': 1.634130525024472,
 'freelancers': 1.410

In [5]:
pdf_path = "pdfs\WHO World health statistics 2025.pdf"

<>:1: SyntaxWarning: invalid escape sequence '\W'
<>:1: SyntaxWarning: invalid escape sequence '\W'
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_28032\2177233425.py:1: SyntaxWarning: invalid escape sequence '\W'
  pdf_path = "pdfs\WHO World health statistics 2025.pdf"


In [6]:
load_dotenv()

True

In [7]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [8]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [65]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)
cur = conn.cursor()

In [10]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()


Loading weights: 100%|██████████| 392/392 [00:00<00:00, 2110.95it/s]
ColQwen2 LOAD REPORT from: vidore/colqwen2-v1.0
Key                                                                    | Status     | 
-----------------------------------------------------------------------+------------+-
base_model.language_model.model.custom_text_proj.lora_A.default.weight | UNEXPECTED | 
base_model.language_model.model.custom_text_proj.lora_B.default.weight | UNEXPECTED | 
custom_text_proj.lora_B.default.weight                                 | MISSING    | 
custom_text_proj.lora_A.default.weight                                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
)

In [12]:
def get_coarse_embedding(embeddings_tensor):
    normalized = embeddings_tensor / embeddings_tensor.norm(dim=-1, keepdim=True)
    return normalized.mean(dim=1).squeeze().to(torch.float32).cpu() 

In [13]:
def embed_query(query, model, processor):
    inputs = processor.process_queries(queries=[query])
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        coarse_vector = get_coarse_embedding(embeddings).flatten().tolist()
        embeddings = embeddings.squeeze(0).to(torch.float32).cpu().numpy()
    return coarse_vector, embeddings


In [13]:
def get_query_weights(query_tokens):
    weights = []
    for t in query_tokens:
        clean_t = t.replace('Ġ', '').lower().strip()
        w = idf_map.get(clean_t, 1.0)
        if any(char.isdigit() for char in clean_t) or '$' in clean_t:
            w *= 1.5
        weights.append(w)
    return np.array(weights, dtype=np.float32)

In [14]:
def coarse_search(coarse_vector, top_k = 50):
    try:
        with conn.cursor() as cur:
            cur.execute('''
                SELECT id, page_number FROM pages
                ORDER BY coarse_vector <=> %s::vector
                LIMIT %s

            ''', (coarse_vector, top_k))
            dic = {}
            for row in cur.fetchall():
                dic[row[0]] = row[1]

            return dic
    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()

In [15]:
def precision_search(query_embeddings, ids, top_k=20):
    try:
        all_page_data = {} 
        with conn.cursor() as cur:
            cur.execute('''
                SELECT page_id, patch_embedding 
                FROM page_patches 
                WHERE page_id IN %s 
                ORDER BY page_id, patch_index
            ''', (tuple(ids),))
            
            rows = cur.fetchall()
            for p_id, emb in rows:
                if p_id not in all_page_data:
                    all_page_data[p_id] = []
                all_page_data[p_id].append(emb)

        results = []
        for p_id, patches in all_page_data.items():
            page_matrix = np.array(patches, dtype=np.float32)
            
            # 1. Similarity Matrix: (Query Tokens x Page Patches)
            sim_matrix = query_embeddings @ page_matrix.T 
            
            # 2. MaxSim: Find best patch for every query token
            token_scores = np.max(sim_matrix, axis=1) 
            
            # 3. Sum: Final scalar score is the unweighted sum of MaxSim scores
            final_score = np.sum(token_scores)
            
            results.append((p_id, float(final_score)))

        results.sort(key=lambda x: x[1], reverse=True)
        return results[:top_k]

    except Exception as e:
        print(f"Database Error: {e}")
        conn.rollback()
        return []

In [25]:
import numpy as np

def hybrid_precision_search(query_text, query_embeddings, candidate_ids, top_k=20):
    """
    Combines ColQwen MaxSim and Postgres FTS using RRF.
    """
    # 0. Safety Check: If no candidates, return empty to avoid SQL 'IN ()' error
    if not candidate_ids:
        return []

    k_rrf = 60  
    rrf_scores = {}

    try:
        with conn.cursor() as cur:
            # --- STEP 1: BM25 / KEYWORD SEARCH ---
            # COALESCE ensures we get 0.0 instead of None if a row has empty fts_tokens
            cur.execute('''
                SELECT id, COALESCE(ts_rank_cd(fts_tokens, plainto_tsquery('english', %s)), 0) AS rank
                FROM pages
                WHERE id IN %s
                ORDER BY rank DESC
            ''', (query_text, tuple(candidate_ids)))
            
            bm25_results = cur.fetchall()
            for rank, (p_id, score) in enumerate(bm25_results):
                # Now score is guaranteed to be a float, avoiding the NoneType error
                if score > 0:
                    rrf_scores[p_id] = rrf_scores.get(p_id, 0) + (1.0 / (k_rrf + rank + 1))

            # --- STEP 2: COLQWEN MAXSIM SEARCH ---
            cur.execute('''
                SELECT page_id, patch_embedding 
                FROM page_patches 
                WHERE page_id IN %s 
                ORDER BY page_id, patch_index
            ''', (tuple(candidate_ids),))
            
            all_page_data = {}
            for p_id, emb in cur.fetchall():
                if p_id not in all_page_data: 
                    all_page_data[p_id] = []
                all_page_data[p_id].append(emb)

            colqwen_list = []
            # Ensure query_embeddings is a numpy array for matrix multiplication
            q_emb = np.array(query_embeddings, dtype=np.float32)
            
            for p_id, patches in all_page_data.items():
                page_matrix = np.array(patches, dtype=np.float32)
                
                # MaxSim calculation: sum of max similarities
                sim_matrix = q_emb @ page_matrix.T 
                final_score = np.sum(np.max(sim_matrix, axis=1))
                colqwen_list.append((p_id, final_score))

            # Sort ColQwen results by score to get their rank
            colqwen_list.sort(key=lambda x: x[1], reverse=True)
            for rank, (p_id, _) in enumerate(colqwen_list):
                rrf_scores[p_id] = rrf_scores.get(p_id, 0) + (1.0 / (k_rrf + rank + 1))

        # --- STEP 3: FINAL FUSION ---
        # Sort by the combined RRF score and return the requested top_k
        final_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        
        return final_results[:top_k]

    except Exception as e:
        print(f"Hybrid Search Error: {e}")
        # Only rollback if the connection is still active and in a transaction
        if conn:
            conn.rollback()
        return []


In [16]:
def retrieve_content(results):
    output = []
    with conn.cursor() as cur:
        for id, score in results:
            cur.execute('''
                SELECT page_number, markdown
                FROM pages
                WHERE id = %s
            ''', (id,))

            row = cur.fetchone()
            output.append({
                "page_number": row[0],
                "markdown": row[1]
            })
    return output

In [17]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load model and tokenizer directly
model_name = 'BAAI/bge-reranker-v2-m3'
tokenizer = AutoTokenizer.from_pretrained(model_name)
reranker = AutoModelForSequenceClassification.from_pretrained(model_name)

reranker.to(device)
reranker.eval()


def rerank(query, content):
    if not content:
        return []

    # 2. Prepare text pairs
    pairs = [[query, item["markdown"]] for item in content]
    
    with torch.no_grad():
        # 3. Tokenize manually - this avoids the 'prepare_for_model' glitch
        inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=8192)
        
        # Move to GPU if available
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        # 4. Get logits (scores)
        scores = reranker(**inputs).logits.view(-1).float()
        
        # Move back to CPU and convert to list
        scores = scores.cpu().tolist()

    # 5. Attach scores and sort
    for i, score in enumerate(scores):
        content[i]["score"] = score

    content.sort(key=lambda x: x["score"], reverse=True)
    return content[:3]


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5265.10it/s]


In [18]:
def get_page_base64(pdf_path, page_number):
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_number - 1)

    zoom = 1.0 
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)

    img_bytes = pix.tobytes("jpg")
    return base64.b64encode(img_bytes).decode('utf-8')


In [101]:
url = "https://api.jina.ai/v1/rerank"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {JINA_API_KEY}"
}

In [125]:
import torch
import numpy as np

def get_query_dependent_context(query_emb, patch_embeddings_list, page_number, grid_w=32, top_k_seeds=3, min_dist=4):
    """
    Finds unique, high-relevance patches by skipping nearby redundant patches.
    """
    # 1. Similarity Math
    d_emb = torch.tensor(patch_embeddings_list).to(query_emb.device)
    sim_matrix = torch.matmul(query_emb, d_emb.T)
    # Get max similarity for each of the 1,024 document patches
    relevance_scores = sim_matrix.max(dim=0).values.cpu().numpy()
    
    # 2. De-duplication Logic (NMS)
    # Sort all patch indices by their score (highest first)
    raw_sorted_indices = np.argsort(relevance_scores)[::-1]
    
    final_seeds = []
    for idx in raw_sorted_indices:
        r, c = divmod(idx, grid_w)
        
        # Check if this patch is too close to any patch we've already picked
        is_too_close = False
        for s_idx in final_seeds:
            sr, sc = divmod(s_idx, grid_w)
            # Manhattan distance (rows + cols apart)
            if (abs(r - sr) + abs(c - sc)) < min_dist:
                is_too_close = True
                break
        
        if not is_too_close:
            final_seeds.append(int(idx))
            
        # Stop once we have our desired number of unique snippets
        if len(final_seeds) >= top_k_seeds:
            break
            
    return {
        "page": page_number,
        "indices": final_seeds, 
        "count": len(final_seeds)
    }


In [156]:
def jina_reranker(query, content):
    candidates = []
    for dic in content:
        page_num = dic["page_number"]
        candidates.append({
            "page": page_num,
            "img": get_page_base64(pdf_path, page_num),
        })
    data = {
        "model": "jina-reranker-m0",
        "query": query,
        "documents": [{"image": c["img"]} for c in candidates]
    }

    response = requests.post(url, headers=headers, json=data)
    results = response.json()
    final = []
    for item in results.get("results", []):
        original_index = item["index"]
        page_data = candidates[original_index]
        page_number = page_data["page"]
        final.append({
            "page_number": page_number,
            "score": item["relevance_score"]
        })

    
    return final

In [48]:
def generate_context(content, pdf_path):
    context = []
    for dic in content:
        context.append({
            "type": "text",
            "text": f"--- Page {dic['page_number']} --- {dic['markdown']}\n"
        })

        b64_img = get_page_base64(pdf_path, dic["page_number"])

        context.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpg;base64,{b64_img}"}
        })
    return HumanMessage(content=context)

In [44]:
def generate_message(query, context):
    messages = [SystemMessage(content="""
    You are a professional assistant. Your goal is to provide accurate answers based ONLY on the provided context which includes the markdown and image of pages of a document.

    RULES:
    1. GROUNDING: If the answer is not contained within the provided context, state clearly that you do not have enough information. DO NOT USE EXTERNAL KNOWLEDGE.
    2. MARKDOWN: Use the markdown text for precise data extractions, quotes and tables.
    3. IMAGES: Use the images to verify layout, check for visual elements, and clarify ambiguous parts of the text.
    4. DISCREPANCY: Treat the page image and markdown as equal independent sources of information. ANY CLAIM OR INFORMATION MUST BE SUPPORTED BY BOTH SOURCES. Flag any discrepancies. 
    5. CITATIONS: You MUST CITE THE SPECIFIC PAGE NUMBER (e.g. [Page 4]) for every fact or claim you make. Each page is labelled at the start (e.g. --- Page 4 ---).
    6. CONTEXT INTEGRATION: Treat the context as a single unified knowledge base, even if they are provided out of chronological order.
    7. RELEVANCE: ONLY USE INFORMATION FROM THE CONTEXT that are relevant to answering the question.  
    8. FORMATTING: Use clear headings and bullet points for complex answers.                          
    """),
    context, 
    HumanMessage(content=f"""
    Question: {query}
    """)

    ]
    return messages

In [21]:
def generate_message(query, context):
    messages = [SystemMessage(content="""
    You are a professional assistant. Your goal is to provide accurate answers based ONLY on the provided context which includes page images of a document.

    RULES:
    1. GROUNDING: If the answer is not contained within the provided context, state clearly that you do not have enough information. DO NOT USE EXTERNAL KNOWLEDGE.
    2. CITATIONS: You MUST CITE THE SPECIFIC PAGE NUMBER (e.g. [Page 4]) for every fact or claim you make. Each page is labelled at the start (e.g. --- Page 4 ---).
    3. CONTEXT INTEGRATION: Treat the context as a single unified knowledge base, even if they are provided out of chronological order.
    4. RELEVANCE: ONLY USE INFORMATION FROM THE CONTEXT that are relevant to answering the question.  
    5. FORMATTING: Use clear headings and bullet points for complex answers.                          
    """),
    context, 
    HumanMessage(content=f"""
    Question: {query}
    """)

    ]
    return messages

TESTING

In [69]:
query = 'What was the specific HALE loss attributed to diabetes mellitus morbidity in the 30–69 age group globally between 2000 and 2019?'


In [70]:
coarse_vector, query_embeddings = embed_query(query, model, processor)
dic = coarse_search(coarse_vector)
ids = [key for key in dic.keys()]
results = hybrid_precision_search(query, query_embeddings, ids)
content = retrieve_content(results)
results = rerank(query, content)
context = generate_context(results, pdf_path)
message = generate_message(query, context)
response = llm.invoke(message).content



In [71]:
response

'The specific HALE loss attributed to diabetes mellitus morbidity in the 30–69 age group globally between 2000 and 2019 was 0.14 years [Page 17].'

In [78]:
print(results[0]["markdown"])

## 1.1 Change in HALE between 2000 and 2019

## 1.1.1 Global

Globally, the 5.4-year increase in HALE at birth between 2000 (58.1 years) and 2019 (63.5 years) was a result of declining mortality and morbidity over time, contributing 5.2 years (96.0%) and 0.2 years (4%), respectively. Reduction in mortality from communicable, maternal, perinatal and nutritional conditions (referred to as communicable diseases) represents the greatest source of gain, contributing a total of 3.4 years to the HALE gain, while mortality reduction from NCDs contributed 1.4 years, and injuries 0.4 years (Fig. 1.2).

By individual cause and broad age groups, the largest gains were made by mortality reduction in preterm birth complications (0.26 years) and birth asphyxia and birth trauma (0.21 years) among infants aged 0-1 years, lower respiratory infections (0.40 years), diarrhoeal diseases (0.35 years) and measles (0.24 years) among children aged 0-4 years, and tuberculosis (0.27 years) and HIV/AIDS (0.22 yea

In [58]:
response

'According to the 2023 distribution data, the South-East Asia Region accounts for 41% of the global population, but it disproportionately bears the highest burden of neglected tropical diseases (NTDs), accounting for 87% of the global number of people needing interventions in 2023 [Page 42]. \n\nHowever, the specific percentage of the global population requiring NTD interventions that resides in the South-East Asia Region is not explicitly stated in the provided text. Therefore, I do not have enough information to provide that percentage.'

In [38]:
df = pd.read_csv('tests/test_set - WHO World health statistics 2025  (2).csv')

In [50]:
questions = df["Question"]

In [51]:
data = {"Question": [], "Response": [], "Page Number": []}
for query in questions:
    coarse_vector, embeddings = embed_query(query, model, processor)
    dic = coarse_search(coarse_vector)
    ids = [key for key in dic.keys()]
    results = precision_search(embeddings, ids)
    content = retrieve_content(results)
    reranked = rerank(query, content)
    context = generate_context(reranked, pdf_path)
    message = generate_message(query, context)
    response = llm.invoke(message).content
    data["Question"].append(query)
    data["Response"].append(response)
    data["Page Number"].append([c["page_number"] for c in reranked])

    
    

In [52]:
df = pd.DataFrame(data)

In [53]:
df

,Question,Response,Page Number
0,Which specific cause provided a HALE gain thro...,The specific cause that provided a HALE gain t...,"[17, 19, 18]"
1,What was the specific HALE loss attributed to ...,The specific HALE loss attributed to diabetes ...,"[17, 19, 13]"
2,Which cause is the leading driver of HALE gain...,The leading driver of HALE gain for the Africa...,"[18, 19, 17]"
3,Which WHO region is the only one to show a HAL...,The only WHO region to show a HALE loss due to...,"[19, 24, 21]"
4,Which condition caused the largest morbidity-r...,The condition that caused the largest morbidit...,"[21, 13, 20]"
5,How many years of HALE were lost in the Region...,The provided context does not specify the numb...,"[21, 20, 13]"
6,What is the leading cause of HALE disadvantage...,The leading cause of the HALE disadvantage for...,"[23, 22, 24]"
7,What is the HALE advantage for females in the ...,"In the Western Pacific Region, stroke is the m...","[24, 22, 23]"
8,What was the global HALE advantage for females...,"In 2021, the global HALE advantage for females...","[23, 22, 24]"
9,Which region saw the smallest female HALE adva...,The region that saw the smallest female HALE a...,"[24, 23, 21]"


In [54]:
df.to_csv('results.csv', index=False)